# Graph End-to-End Validation Notebook
This notebook now executes the example modules in sequence, reusing objects across steps:
1. Load `.env` and import reusable `run_*` functions from `examples/`.
2. Run site/auth discovery and capture shared context (`client`, `drive`, `list_client`).
3. Run drive examples in order (list, upload, read/write, download).
4. Run list examples in order (get, create, update).

In [ ]:
from pathlib import Path
from typing import Any, cast
import importlib
import sys
import os

import pandas as pd
from dotenv import load_dotenv

from msgraphclient.auth import GraphClient
from msgraphclient.drive import GraphDrive
from msgraphclient.lists import GraphList

# Ensure .env is loaded from repository root.
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

# Ensure repository root is importable for examples package modules.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


# Helper functio to test a dataframe if it has data and it is not too wide to display in full.
def display_dataframe(df: pd.DataFrame) -> None:
    print(f"\nItems currently loaded: {len(df)}")
    if df.empty:
        print("No items loaded into DataFrame.")
    elif len(df.columns) == 0:
        print("DataFrame has no columns.")
    elif len(df.columns) < 50:
        display(df)
    else:
        print("DataFrame has too many columns to display.")
        display(df.head())

# Import and reload so notebook reruns always pick up latest local edits.
site_mod = importlib.reload(importlib.import_module('examples.example_site_contents'))
drive_list_mod = importlib.reload(importlib.import_module('examples.example_drive_list'))
drive_upload_mod = importlib.reload(importlib.import_module('examples.example_drive_upload'))
drive_rw_mod = importlib.reload(importlib.import_module('examples.example_drive_read_write'))
drive_download_mod = importlib.reload(importlib.import_module('examples.example_drive_download'))
list_get_mod = importlib.reload(importlib.import_module('examples.example_list_get'))
list_create_mod = importlib.reload(importlib.import_module('examples.example_list_create'))
list_update_mod = importlib.reload(importlib.import_module('examples.example_list_update'))

run_example_site_contents = site_mod.run_example_site_contents
run_example_drive_list = drive_list_mod.run_example_drive_list
run_example_drive_upload = drive_upload_mod.run_example_drive_upload
run_example_drive_read_write = drive_rw_mod.run_example_drive_read_write
run_example_drive_download = drive_download_mod.run_example_drive_download
run_example_list_get = list_get_mod.run_example_list_get
run_example_list_create = list_create_mod.run_example_list_create
run_example_list_update = list_update_mod.run_example_list_update

ctx: dict[str, Any] = {}

print(f'.env loaded from: {env_path}')
print('Ready to execute example modules in sequence.')

In [ ]:
ctx.update(run_example_site_contents(show_output=True))

client = cast(GraphClient, ctx['client'])
drive = cast(GraphDrive | None, ctx.get('drive'))
list_client = cast(GraphList | None, ctx.get('list_client'))

site_attributes = {
    'sharepoint_site_id': client.authenticator.sharepoint_site_id,
    'site_graph_id': client.site_graph_id,
    'site_name': client.site_name,
    'site_display_name': client.site_display_name,
    'site_web_url': client.site_web_url,
}

print('\nConnected site attributes:')
for key, value in site_attributes.items():
    print(f'- {key}: {value}')

In [ ]:
site_contents = ctx.get('site_contents', {})
site_data = site_contents.get('site', {})
drives_data = site_contents.get('drives', [])
lists_data = site_contents.get('lists', [])

print('Site contents summary:')
print(f"- Site: {site_data.get('displayName', site_data.get('name', '-'))}")
print(f"- Drives found: {len(drives_data)}")
print(f"- Lists found: {len(lists_data)}")

df_site_drives = pd.json_normalize(drives_data) if drives_data else pd.DataFrame()
df_site_lists = pd.json_normalize(lists_data) if lists_data else pd.DataFrame()

print('\nAvailable Drives:')
display_dataframe(df_site_drives)

print('Available Lists:')
display_dataframe(df_site_lists)

In [ ]:
drive_context = run_example_drive_list(
    client=cast(GraphClient | None, ctx.get('client')),
    drive=cast(GraphDrive | None, ctx.get('drive')),
    folder_path='root',
    show_output=False,
 )
ctx.update(drive_context)

drive_items = cast(list[dict[str, Any]], drive_context.get('drive_items', []))
df_drive_items = pd.json_normalize(drive_items) if drive_items else pd.DataFrame()

num_files = len([item for item in drive_items if 'folder' not in item])
print(f'\nTotal items in drive root: {len(drive_items)}')
print(f'Total files (excluding folders): {num_files}')
display_dataframe(df_drive_items)

## Drive Examples In Sequence
This block calls the reusable drive examples, reusing `client` and `drive` from the shared context.

In [ ]:
test_local_file = repo_root / 'notebooks' / 'downloads' / 'notebook_test_upload.txt'

upload_context = run_example_drive_upload(
    client=cast(GraphClient | None, ctx.get('client')),
    drive=cast(GraphDrive | None, ctx.get('drive')),
    local_file=test_local_file,
    remote_folder='root',
    show_output=True,
 )
ctx.update(upload_context)

uploaded_item_id = ''
if upload_context.get('upload_result'):
    uploaded_item_id = str(upload_context['upload_result'].get('id', ''))

read_write_target_id = uploaded_item_id or os.getenv('DRIVE_ITEM_ID', '').strip()
read_write_context = run_example_drive_read_write(
    client=cast(GraphClient | None, ctx.get('client')),
    drive=cast(GraphDrive | None, ctx.get('drive')),
    item_id=read_write_target_id,
    show_output=True,
 )
ctx.update(read_write_context)

download_context = run_example_drive_download(
    client=cast(GraphClient | None, ctx.get('client')),
    drive=cast(GraphDrive | None, ctx.get('drive')),
    item_id=uploaded_item_id,
    local_folder=repo_root / 'notebooks' / 'downloads',
    show_output=True,
 )
ctx.update(download_context)

## List Examples In Sequence
This block calls list examples directly, reusing `client` and `list_client` from the shared context.

In [ ]:
list_get_context = run_example_list_get(
    client=cast(GraphClient | None, ctx.get('client')),
    list_client=cast(GraphList | None, ctx.get('list_client')),
    interactive=False,
    show_output=False,
 )
ctx.update(list_get_context)

df_list_content = cast(pd.DataFrame, list_get_context.get('df_list_content', pd.DataFrame()))
print(f'\nItems currently loaded: {len(df_list_content)}')

display_dataframe(df_list_content)

list_client = cast(GraphList | None, ctx.get('list_client'))
if list_client is not None:
    schema = list_client.get_schema()
    field_types = list_client.get_field_types()
    print(f'\nSchema columns available: {len(schema)}')
    print(f'Field type mapping entries: {len(field_types)}')
    display_dataframe(pd.DataFrame(schema))

### Save Operations Via Examples
This block invokes create and update examples in sequence, reusing the same `list_client`.
The update example will target the first available item if no explicit item ID is provided.

In [ ]:
create_context = run_example_list_create(
    client=cast(GraphClient | None, ctx.get('client')),
    list_client=cast(GraphList | None, ctx.get('list_client')),
    item_fields={'Title': 'Notebook create example item'},
    show_output=True,
 )
ctx.update(create_context)

created_item_id = ''
if create_context.get('created_item'):
    created_item_id = str(create_context['created_item'].get('_id', '')).strip()

update_context = run_example_list_update(
    client=cast(GraphClient | None, ctx.get('client')),
    list_client=cast(GraphList | None, ctx.get('list_client')),
    item_id=created_item_id,
    show_output=True,
 )
ctx.update(update_context)

final_list_state = run_example_list_get(
    client=cast(GraphClient | None, ctx.get('client')),
    list_client=cast(GraphList | None, ctx.get('list_client')),
    interactive=False,
    show_output=False,
 )

print('\nUpdated list sample after create + update:')
display_dataframe(
    cast(pd.DataFrame, final_list_state.get("df_list_content", pd.DataFrame())))